In [2]:
import os

print(os.listdir('/kaggle/input'))

['notebooks', 'datasets', 'competitions']


In [3]:
print(os.listdir('/kaggle/input/datasets'))

['mayankptdr']


In [4]:
print(os.listdir('/kaggle/input/datasets/mayankptdr'))

['rsna-3', 'rsna-1', 'rsna-4', 'rsna-2', 'rsna-5', 'report-labels']


In [5]:
for name in ['rsna-1', 'rsna-2', 'rsna-3', 'rsna-4', 'rsna-5']:
    path = f'/kaggle/input/datasets/mayankptdr/{name}'
    print(f"=== {name} ===")
    print(os.listdir(path))

=== rsna-1 ===
['cached_volumes']
=== rsna-2 ===
['cached_volumes']
=== rsna-3 ===
['cached_volumes']
=== rsna-4 ===
['cached_volumes']
=== rsna-5 ===
['cached_volumes']


In [6]:
import os

base = '/kaggle/input/datasets/mayankptdr'

cache_dirs = []
total_files = 0

for name in ['rsna-1', 'rsna-2', 'rsna-3', 'rsna-4', 'rsna-5']:
    path = f"{base}/{name}/cached_volumes"
    count = len(os.listdir(path))
    print(f"{name}: {count} files")
    cache_dirs.append(path)
    total_files += count

print(f"\nTotal cached studies: {total_files}")

rsna-1: 881 files
rsna-2: 881 files
rsna-3: 879 files
rsna-4: 881 files
rsna-5: 883 files

Total cached studies: 4405


In [7]:
import pandas as pd

data_dir = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

train = pd.read_csv(f"{data_dir}/train.csv")

finding_columns = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 
                    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 
                    'Contusion', 'Fracture']

gold_data = train[train['Fracture'].notna()]
gold_labels = gold_data[['StudyInstanceUID'] + finding_columns].copy()

report_label = pd.read_csv("/kaggle/input/datasets/mayankptdr/report-labels/extracted_labels.csv")

all_labels = pd.concat([gold_labels, report_label], ignore_index=True)
all_labels[finding_columns] = all_labels[finding_columns].fillna(0)

print(all_labels.shape)

(4407, 13)


In [8]:
import torch
from torch.utils.data import Dataset
import numpy as np

class KneeMRIDatasetCached(Dataset):
    def __init__(self, labels_df, cache_dirs):
        self.labels_df = labels_df.reset_index(drop=True)
        self.cache_dirs = cache_dirs
        self.finding_columns = finding_columns
        
        self.valid_indices = []
        for idx, row in self.labels_df.iterrows():
            if self._find_cache_path(row['StudyInstanceUID']) is not None:
                self.valid_indices.append(idx)
        
        print(f"Valid studies (cache found): {len(self.valid_indices)} / {len(self.labels_df)}")
    
    def _find_cache_path(self, study_id):
        for d in self.cache_dirs:
            path = f"{d}/{study_id}.npz"
            if os.path.exists(path):
                return path
        return None
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, i):
        idx = self.valid_indices[i]
        row = self.labels_df.iloc[idx]
        study_id = row['StudyInstanceUID']
        
        cache_path = self._find_cache_path(study_id)
        data = np.load(cache_path)
        
        sagittal = torch.tensor(data['sagittal'], dtype=torch.float32)
        coronal = torch.tensor(data['coronal'], dtype=torch.float32)
        axial = torch.tensor(data['axial'], dtype=torch.float32)
        
        labels = torch.tensor(row[self.finding_columns].values.astype(float), dtype=torch.float32)
        
        return {'sagittal': sagittal, 'coronal': coronal, 'axial': axial, 
                'labels': labels, 'study_id': study_id}

dataset = KneeMRIDatasetCached(all_labels, cache_dirs)
sample = dataset[0]
print(sample['sagittal'].shape, sample['labels'])

Valid studies (cache found): 4405 / 4407
torch.Size([24, 256, 256]) tensor([0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0.])


In [9]:
!pip install -q iterative-stratification

In [10]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
import numpy as np

# sirf valid studies (jinki cache hai) use karo
valid_labels = all_labels.iloc[dataset.valid_indices].reset_index(drop=True)

X = valid_labels['StudyInstanceUID'].values
y = valid_labels[finding_columns].values

mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

folds = []
for fold_num, (train_idx, val_idx) in enumerate(mskf.split(X, y)):
    folds.append({'train_idx': train_idx, 'val_idx': val_idx})
    print(f"Fold {fold_num}: train={len(train_idx)}, val={len(val_idx)}")

Fold 0: train=3524, val=881
Fold 1: train=3524, val=881
Fold 2: train=3524, val=881
Fold 3: train=3524, val=881
Fold 4: train=3524, val=881


In [11]:
import torch
import torch.nn as nn
import torchvision

class SinglePlaneEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        base = torchvision.models.video.r3d_18(weights='KINETICS400_V1' if pretrained else None)
        
        # first-conv layer 1st channel
        old_conv = base.stem[0]
        new_conv = nn.Conv3d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                              stride=old_conv.stride, padding=old_conv.padding, bias=False)
        if pretrained:
            new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
        base.stem[0] = new_conv
        
        base.fc = nn.Identity()
        self.encoder = base
    
    def forward(self, x):
        x = x.unsqueeze(1)  
        return self.encoder(x)  

class MultiPlaneKneeModel(nn.Module):
    def __init__(self, num_findings=12, pretrained=True):
        super().__init__()
        self.sagittal_enc = SinglePlaneEncoder(pretrained)
        self.coronal_enc = SinglePlaneEncoder(pretrained)
        self.axial_enc = SinglePlaneEncoder(pretrained)
        
        self.head = nn.Sequential(
            nn.Linear(512 * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_findings)
        )
    
    def forward(self, sagittal, coronal, axial):
        s = self.sagittal_enc(sagittal)
        c = self.coronal_enc(coronal)
        a = self.axial_enc(axial)
        combined = torch.cat([s, c, a], dim=1)
        return self.head(combined)

In [12]:
model = MultiPlaneKneeModel(num_findings=12, pretrained=True)

from torch.utils.data import DataLoader
loader = DataLoader(dataset, batch_size=2, shuffle=True)
batch = next(iter(loader))

output = model(batch['sagittal'], batch['coronal'], batch['axial'])
print(output.shape)  # (2, 12) expect karo

Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:00<00:00, 197MB/s]  


torch.Size([2, 12])


In [13]:
import numpy as np

pos_counts = valid_labels[finding_columns].sum().values
neg_counts = len(valid_labels) - pos_counts
pos_weights = neg_counts / (pos_counts + 1e-6)  # rare-findings ko zyada weight
pos_weights = torch.tensor(pos_weights, dtype=torch.float32)

print(dict(zip(finding_columns, pos_weights.numpy().round(2))))

{'ACL': np.float32(4.22), 'MCL': np.float32(5.86), 'Medial Meniscus': np.float32(1.34), 'Lateral Meniscus': np.float32(4.92), 'Medial OA': np.float32(1.76), 'Lateral OA': np.float32(3.73), 'PF OA': np.float32(1.23), 'Effusion': np.float32(0.66), 'Synovitis': np.float32(6.66), "Baker's": np.float32(3.03), 'Contusion': np.float32(3.14), 'Fracture': np.float32(11.81)}


In [14]:
# Training Loop
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

checkpoint_dir = "/kaggle/working/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

def train_one_fold(fold_num, train_idx, val_idx, epochs=10, batch_size=4, lr=1e-4):
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)
    
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    model = MultiPlaneKneeModel(num_findings=12, pretrained=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(device))
    
    checkpoint_path = f"{checkpoint_dir}/fold{fold_num}_latest.pt"
    start_epoch = 0
    best_auroc = 0
    
    # resume-logic
    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_epoch = ckpt['epoch'] + 1
        best_auroc = ckpt['best_auroc']
        print(f"Resumed fold {fold_num} from epoch {start_epoch}")
    
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            sagittal = batch['sagittal'].to(device)
            coronal = batch['coronal'].to(device)
            axial = batch['axial'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(sagittal, coronal, axial)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # validation
        model.eval()
        all_preds, all_labels_val = [], []
        with torch.no_grad():
            for batch in val_loader:
                sagittal = batch['sagittal'].to(device)
                coronal = batch['coronal'].to(device)
                axial = batch['axial'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = torch.sigmoid(model(sagittal, coronal, axial))
                all_preds.append(outputs.cpu().numpy())
                all_labels_val.append(labels.cpu().numpy())
        
        all_preds = np.concatenate(all_preds)
        all_labels_val = np.concatenate(all_labels_val)
        
        aurocs = []
        for i in range(12):
            try:
                auroc = roc_auc_score(all_labels_val[:, i], all_preds[:, i])
                aurocs.append(auroc)
            except ValueError:
                aurocs.append(0.5)  # agar ek-hi-class ho fold mein, skip-safe
        
        macro_auroc = np.mean(aurocs)
        avg_train_loss = train_loss / len(train_loader)
        
        print(f"Fold {fold_num} | Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val AUROC: {macro_auroc:.4f}")
        
        is_best = macro_auroc > best_auroc
        best_auroc = max(best_auroc, macro_auroc)
        
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_auroc': best_auroc,
        }, checkpoint_path)
        
        if is_best:
            torch.save(model.state_dict(), f"{checkpoint_dir}/fold{fold_num}_best.pt")
    
    return best_auroc

Using device: cuda


In [15]:
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

checkpoint_dir = "/kaggle/working/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

def train_one_fold(fold_num, train_idx, val_idx, epochs=10, batch_size=4, lr=1e-4):
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)
    
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    model = MultiPlaneKneeModel(num_findings=12, pretrained=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(device))
    scaler = GradScaler('cuda')
    
    checkpoint_path = f"{checkpoint_dir}/fold{fold_num}_latest.pt"
    start_epoch = 0
    best_auroc = 0
    
    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_epoch = ckpt['epoch'] + 1
        best_auroc = ckpt['best_auroc']
        print(f"Resumed fold {fold_num} from epoch {start_epoch}")
    
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            sagittal = batch['sagittal'].to(device)
            coronal = batch['coronal'].to(device)
            axial = batch['axial'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(sagittal, coronal, axial)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
        
        model.eval()
        all_preds, all_labels_val = [], []
        with torch.no_grad():
            for batch in val_loader:
                sagittal = batch['sagittal'].to(device)
                coronal = batch['coronal'].to(device)
                axial = batch['axial'].to(device)
                labels = batch['labels'].to(device)
                
                with autocast('cuda'):
                    outputs = torch.sigmoid(model(sagittal, coronal, axial))
                all_preds.append(outputs.cpu().numpy())
                all_labels_val.append(labels.cpu().numpy())
        
        all_preds = np.concatenate(all_preds)
        all_labels_val = np.concatenate(all_labels_val)
        
        aurocs = []
        for i in range(12):
            try:
                auroc = roc_auc_score(all_labels_val[:, i], all_preds[:, i])
                aurocs.append(auroc)
            except ValueError:
                aurocs.append(0.5)
        
        macro_auroc = np.mean(aurocs)
        avg_train_loss = train_loss / len(train_loader)
        print(f"Fold {fold_num} | Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val AUROC: {macro_auroc:.4f}")
        
        is_best = macro_auroc > best_auroc
        best_auroc = max(best_auroc, macro_auroc)
        
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_auroc': best_auroc,
        }, checkpoint_path)
        
        if is_best:
            torch.save(model.state_dict(), f"{checkpoint_dir}/fold{fold_num}_best.pt")
    
    return best_auroc

Using device: cuda


In [ ]:
fold_results = {}
fold_results[0] = 0.7155

for fold_num in range(1, 5):
    print(f"\n{'='*50}")
    print(f"Starting Fold {fold_num}")
    print('='*50)
    
    best_auroc = train_one_fold(
        fold_num=fold_num,
        train_idx=folds[fold_num]['train_idx'],
        val_idx=folds[fold_num]['val_idx'],
        epochs=8,
        batch_size=4
    )
    
    fold_results[fold_num] = best_auroc
    print(f"Fold {fold_num} complete. Best AUROC: {best_auroc:.4f}")
print(f"\n{'='*50}")
print("All folds complete!")
print(fold_results)
print(f"Average AUROC: {np.mean(list(fold_results.values())):.4f}")


Starting Fold 1
Fold 1 | Epoch 1/8 | Train Loss: 0.9525 | Val AUROC: 0.7091
Fold 1 | Epoch 2/8 | Train Loss: 0.9135 | Val AUROC: 0.7546
Fold 1 | Epoch 3/8 | Train Loss: 0.8704 | Val AUROC: 0.7650
Fold 1 | Epoch 4/8 | Train Loss: 0.8294 | Val AUROC: 0.7842
Fold 1 | Epoch 5/8 | Train Loss: 0.7857 | Val AUROC: 0.7956
Fold 1 | Epoch 6/8 | Train Loss: 0.7431 | Val AUROC: 0.8090
Fold 1 | Epoch 7/8 | Train Loss: 0.6866 | Val AUROC: 0.8049
Fold 1 | Epoch 8/8 | Train Loss: 0.6399 | Val AUROC: 0.8070
Fold 1 complete. Best AUROC: 0.8090

Starting Fold 2
Fold 2 | Epoch 1/8 | Train Loss: 0.9696 | Val AUROC: 0.7226
Fold 2 | Epoch 2/8 | Train Loss: 0.9147 | Val AUROC: 0.7487
Fold 2 | Epoch 3/8 | Train Loss: 0.8714 | Val AUROC: 0.7769
Fold 2 | Epoch 4/8 | Train Loss: 0.8310 | Val AUROC: 0.7945
Fold 2 | Epoch 5/8 | Train Loss: 0.7918 | Val AUROC: 0.8025


In [16]:
import shutil
import os

old_ckpt_dir = '/kaggle/input/notebooks/mayankptdr/rsna-training/checkpoints'
checkpoint_dir = '/kaggle/working/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

for f in os.listdir(old_ckpt_dir):
    shutil.copy(f"{old_ckpt_dir}/{f}", f"{checkpoint_dir}/{f}")

print("Restored:", os.listdir(checkpoint_dir))

Restored: ['fold3_best.pt', 'fold1_best.pt', 'fold0_best.pt', 'fold4_latest.pt', 'fold3_latest.pt', 'fold4_best.pt', 'fold1_latest.pt', 'fold2_latest.pt', 'fold0_latest.pt', 'fold2_best.pt']


In [17]:
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

checkpoint_dir = "/kaggle/working/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

def train_one_fold(fold_num, train_idx, val_idx, epochs=10, batch_size=2, lr=1e-4):
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)
    
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    model = MultiPlaneKneeModel(num_findings=12, pretrained=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(device))
    scaler = GradScaler('cuda')
    
    checkpoint_path = f"{checkpoint_dir}/fold{fold_num}_latest.pt"
    start_epoch = 0
    best_auroc = 0
    
    # RESUME-LOGIC: if checkpoint exist the traing will be cintinue from checkpoint
    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_epoch = ckpt['epoch'] + 1
        best_auroc = ckpt['best_auroc']
        print(f"Resumed fold {fold_num} from epoch {start_epoch}")
    
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            sagittal = batch['sagittal'].to(device)
            coronal = batch['coronal'].to(device)
            axial = batch['axial'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(sagittal, coronal, axial)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
        
        model.eval()
        all_preds, all_labels_val = [], []
        with torch.no_grad():
            for batch in val_loader:
                sagittal = batch['sagittal'].to(device)
                coronal = batch['coronal'].to(device)
                axial = batch['axial'].to(device)
                labels = batch['labels'].to(device)
                
                with autocast('cuda'):
                    outputs = torch.sigmoid(model(sagittal, coronal, axial))
                all_preds.append(outputs.cpu().numpy())
                all_labels_val.append(labels.cpu().numpy())
        
        all_preds = np.concatenate(all_preds)
        all_labels_val = np.concatenate(all_labels_val)
        
        aurocs = []
        for i in range(12):
            try:
                auroc = roc_auc_score(all_labels_val[:, i], all_preds[:, i])
                aurocs.append(auroc)
            except ValueError:
                aurocs.append(0.5)
        
        macro_auroc = np.mean(aurocs)
        avg_train_loss = train_loss / len(train_loader)
        print(f"Fold {fold_num} | Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val AUROC: {macro_auroc:.4f}")
        
        is_best = macro_auroc > best_auroc
        best_auroc = max(best_auroc, macro_auroc)
        
        # === YEH CHECKPOINT-SAVE-KARTA-HAI, HAR-EPOCH-KE-BAAD ===
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_auroc': best_auroc,
        }, checkpoint_path)
        
        if is_best:
            torch.save(model.state_dict(), f"{checkpoint_dir}/fold{fold_num}_best.pt")
        # =========================================================
    
    return best_auroc

Using device: cuda


In [ ]:
# Training continue with checkpoints
fold_results = {}
fold_results[0] = 0.7155
fold_results[1] = 0.8090

for fold_num in range(2, 5):
    print(f"\n{'='*50}")
    print(f"Starting Fold {fold_num}")
    print('='*50)
    
    best_auroc = train_one_fold(
        fold_num=fold_num,
        train_idx=folds[fold_num]['train_idx'],
        val_idx=folds[fold_num]['val_idx'],
        epochs=8,
        batch_size=2
    )
    
    fold_results[fold_num] = best_auroc
    print(f"Fold {fold_num} complete. Best AUROC: {best_auroc:.4f}")

print(f"\n{'='*50}")
print("All folds complete!")
print(fold_results)
print(f"Average AUROC: {np.mean(list(fold_results.values())):.4f}")


Starting Fold 2
Resumed fold 2 from epoch 8
Fold 2 complete. Best AUROC: 0.7118

Starting Fold 3
Fold 3 | Epoch 1/8 | Train Loss: 0.9902 | Val AUROC: 0.6586
Fold 3 | Epoch 2/8 | Train Loss: 0.9583 | Val AUROC: 0.6646
Fold 3 | Epoch 3/8 | Train Loss: 0.9342 | Val AUROC: 0.6847
Fold 3 | Epoch 4/8 | Train Loss: 0.9150 | Val AUROC: 0.7027
Fold 3 | Epoch 5/8 | Train Loss: 0.8936 | Val AUROC: 0.6964
Fold 3 | Epoch 6/8 | Train Loss: 0.8727 | Val AUROC: 0.7000
Fold 3 | Epoch 7/8 | Train Loss: 0.8494 | Val AUROC: 0.7045
Fold 3 | Epoch 8/8 | Train Loss: 0.8190 | Val AUROC: 0.6992
Fold 3 complete. Best AUROC: 0.7045

Starting Fold 4
Fold 4 | Epoch 1/8 | Train Loss: 1.0049 | Val AUROC: 0.6363
Fold 4 | Epoch 2/8 | Train Loss: 0.9844 | Val AUROC: 0.6733
Fold 4 | Epoch 3/8 | Train Loss: 0.9738 | Val AUROC: 0.6847


In [18]:
# Training continue with checkpoints
fold_results = {}
fold_results[0] = 0.7155
fold_results[1] = 0.8090

for fold_num in range(2, 5):
    print(f"\n{'='*50}")
    print(f"Starting Fold {fold_num}")
    print('='*50)
    
    best_auroc = train_one_fold(
        fold_num=fold_num,
        train_idx=folds[fold_num]['train_idx'],
        val_idx=folds[fold_num]['val_idx'],
        epochs=8,
        batch_size=2
    )
    
    fold_results[fold_num] = best_auroc
    print(f"Fold {fold_num} complete. Best AUROC: {best_auroc:.4f}")

print(f"\n{'='*50}")
print("All folds complete!")
print(fold_results)
print(f"Average AUROC: {np.mean(list(fold_results.values())):.4f}")


Starting Fold 2
Resumed fold 2 from epoch 8
Fold 2 complete. Best AUROC: 0.7118

Starting Fold 3
Resumed fold 3 from epoch 8
Fold 3 complete. Best AUROC: 0.7045

Starting Fold 4
Resumed fold 4 from epoch 3


KeyboardInterrupt: 

In [22]:
models = []
for fold_num in range(5):
    m = MultiPlaneKneeModel(num_findings=12, pretrained=False).to(device)
    state = torch.load(f"{checkpoint_dir}/fold{fold_num}_best.pt", map_location=device, weights_only=False)
    m.load_state_dict(state)
    m.eval()
    models.append(m)

print(f"Loaded {len(models)} models for ensembling")

from scipy.ndimage import zoom

def resize_volume(volume, target_shape=(24, 256, 256)):
    factors = [t/s for t, s in zip(target_shape, volume.shape)]
    resized = zoom(volume, factors, order=1)
    return resized.astype(np.float32)

Loaded 5 models for ensembling


In [23]:
test_data_dir = f"{data_dir}/test_series"
test_series_csv = pd.read_csv(f"{data_dir}/test_series.csv")
test_csv = pd.read_csv(f"{data_dir}/test.csv")

def select_series_test(study_id):
    study_series = test_series_csv[test_series_csv['StudyInstanceUID'] == study_id]
    selected = {}
    for plane in ['Sagittal', 'Coronal', 'Axial']:
        plane_series = study_series[study_series['Anatomical_Plane'] == plane]
        chosen = plane_series[plane_series['Fluid_Sensitive'] == 1]
        if len(chosen) == 0:
            chosen = plane_series
        selected[plane] = chosen.iloc[0]['SeriesInstanceUID']
    return selected

def load_study_test(study_id, target_shape=(24, 256, 256)):
    series_map = select_series_test(study_id)
    volumes = {}
    for plane, series_id in series_map.items():
        folder = f"{data_dir}/test_series/{study_id}/{series_id}"
        dcm_files = os.listdir(folder)
        slices = [pydicom.dcmread(f"{folder}/{f}") for f in dcm_files]
        slices_sorted = sorted(slices, key=lambda d: int(d.InstanceNumber))
        volume = np.stack([s.pixel_array for s in slices_sorted])
        volume_norm = (volume - volume.min()) / (volume.max() - volume.min())
        volume_norm = volume_norm.astype(np.float32)
        volume_resized = resize_volume(volume_norm, target_shape)
        volumes[plane] = volume_resized
    return volumes

def predict_study(study_id):
    volumes = load_study_test(study_id)
    sagittal = torch.tensor(volumes['Sagittal'], dtype=torch.float32).unsqueeze(0).to(device)
    coronal = torch.tensor(volumes['Coronal'], dtype=torch.float32).unsqueeze(0).to(device)
    axial = torch.tensor(volumes['Axial'], dtype=torch.float32).unsqueeze(0).to(device)
    
    all_probs = []
    with torch.no_grad():
        for m in models:
            with autocast('cuda'):
                out = torch.sigmoid(m(sagittal, coronal, axial))
            all_probs.append(out.cpu().numpy())
    
    avg_probs = np.mean(all_probs, axis=0)[0]
    return avg_probs

In [24]:
import pydicom

predictions = []
for study_id in test_csv['StudyInstanceUID']:
    try:
        probs = predict_study(study_id)
        predictions.append([study_id] + list(probs))
        print(f"Predicted: {study_id}")
    except Exception as e:
        print(f"Error on {study_id}: {e}")
        predictions.append([study_id] + [0.5]*12)  # fallback

submission = pd.DataFrame(predictions, columns=['StudyInstanceUID'] + finding_columns)
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head())

Predicted: 1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191
Predicted: 1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776
Predicted: 1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.413574  0.430420   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.499268  0.516602   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.465088  0.478760   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.437012          0.376221   0.435791    0.450439  0.491943   
1         0.522949          0.487061   0.545898    0.531250  0.541992   
2         0.483887          0.427246   0.454590    0.480225  0.514648   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.432129   0.287109  0.459717   0.414062  0.342773  
1  0.576660   0.471680  0.530273   0.502930  0.473633  
2  0.5

/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [25]:
sample_sub = pd.read_csv(f"{data_dir}/sample_submission.csv")
print(sample_sub.columns.tolist())
print(sample_sub.shape)

print(submission.columns.tolist())
print(submission.shape)

['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
(3, 13)
['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
(3, 13)
